# 01 · Computer Vision with Transformers
### Seeing the world — for biodiversity and environmental monitoring

AI image models are already used by communities and researchers to **identify
species**, **monitor ecosystems**, read **camera-trap** photos, and flag plant
diseases from a phone photo. In this notebook we use **Hugging Face** vision
models to do exactly this kind of task.

But "seeing" is never neutral. A vision model can only recognise what it was
trained to recognise. If its training images came mostly from one part of the
world, it will be confident about *those* plants, animals and landscapes — and
blind to others. So alongside the code we keep asking:

> **Whose images trained this model? Whose categories does it use? Who is it
> built to see — and who does it overlook?**

We will use a small public dataset of **bean leaves** (healthy vs. two
diseases).


## How to use this notebook

This is **Workshop 3** of the *Introduction to AI* series. Workshops 1 and 2
looked at tabular data, images, tokenization and retrieval. Workshop 3 is a
hands-on tour of the **Hugging Face** ecosystem across three applications.

The notebooks are designed to be run in **Google Colab**, top to bottom.

Each notebook is organised in three levels:

- 🟢 **Beginner** — run pretrained models with a few lines of code.
- 🟡 **Medium** — adapt models, change parameters, and read the results critically.
- 🔴 **Optional / Advanced** — train or fine-tune a model yourself (best with a GPU).

Look out for these blocks:

- `# TODO:` cells are **your turn to code**. Try them before scrolling to the solution.
- **▶️ Play with it** cells are safe to edit and rerun — change models, labels, or numbers.
- **🧭 Critical reflection** boxes connect the technical work to the bigger questions of
  the workshop: *who designs these systems, who do they serve, whose knowledge do they
  include, and whose values do they encode?*

> 💡 **Before you start:** switch on a GPU via
> `Runtime -> Change runtime type -> Hardware accelerator -> GPU`.
> Everything still runs on CPU, just more slowly.

Workshop 3 sequence:
1. `01_computer_vision_with_transformers.ipynb`
2. `02_finetuning_transformers_for_text.ipynb`
3. `03_transformers_for_sensor_time_series.ipynb`


### 🧭 Why vision models matter for communities

- **Biodiversity & conservation:** identifying species from photos or camera traps.
- **Environmental monitoring:** spotting deforestation, algae blooms, crop disease.
- **Dual-use warning:** the same "detect and classify" technology powers **surveillance**. Keep asking who benefits and who is watched.


In [ ]:
# --- Self-contained runtime setup -------------------------------------------
# Every notebook sets its own seed and device so it can run independently.

import random
import sys
from pathlib import Path

import numpy as np

try:
    import torch
except ImportError:
    torch = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"
# Hugging Face pipelines expect device=0 for the first GPU, or device=-1 for CPU.
PIPELINE_DEVICE = 0 if DEVICE == "cuda" else -1

print("Python version:", sys.version.split()[0])
print("Detected device:", DEVICE)
if DEVICE == "cpu":
    print("Tip: enable a GPU via Runtime -> Change runtime type for faster runs.")
print("Seed set to:", SEED)

NOTEBOOK_FRAMING = "computer vision for biodiversity and environmental monitoring"
NOTEBOOK_FRAMING


In [ ]:
# In Colab these are usually already installed. Uncomment if an import fails.
# !pip -q install transformers datasets pillow matplotlib

print("Vision notebook ready once transformers, datasets, pillow and matplotlib are available.")


## 🟢 Part 1 — Run a pretrained image classifier (the easy way)

The fastest way to use a model on Hugging Face is the **`pipeline`**. You pick a
task and a model name, and it handles downloading, preprocessing and inference.

We start with **ViT** (a Vision Transformer) trained on **ImageNet** — a famous
dataset of ~1.2 million web images across 1,000 everyday categories.


In [ ]:
from transformers import pipeline

# "image-classification" + an ImageNet-trained ViT.
# ViT = Vision Transformer: it splits an image into patches and treats them
# like a sequence of tokens, the same idea transformers use for text.
imagenet_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224",
    device=PIPELINE_DEVICE,
)

print("Loaded:", imagenet_classifier.model.name_or_path)
print("This model knows", imagenet_classifier.model.config.num_labels, "categories.")


In [ ]:
# Load a few real leaf photos from the public "beans" dataset.
# (Small dataset of bean leaves: healthy, angular leaf spot, bean rust.)
from datasets import load_dataset

beans = load_dataset("AI-Lab-Makerere/beans", split="validation")
class_names = beans.features["labels"].names

print("Number of images:", len(beans))
print("True categories in this dataset:", class_names)


In [ ]:
import matplotlib.pyplot as plt

# Show one example image with its TRUE (human-given) label.
example = beans[0]
image = example["image"]

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis("off")
plt.title("True label: " + class_names[example["labels"]])
plt.show()


In [ ]:
# Ask the ImageNet model what it thinks this leaf is.
predictions = imagenet_classifier(image, top_k=5)

print("Top-5 guesses from an ImageNet-trained model:")
for rank, p in enumerate(predictions, start=1):
    print(f"  {rank}. {p['label']:<30} score={p['score']:.3f}")


### 🧭 Critical reflection: notice the mismatch

The model is *fluent and confident* — but its 1,000 categories were chosen by
the people who built ImageNet. "Angular leaf spot on a bean plant" is simply not
in its vocabulary, so it reaches for the nearest web-photo category it knows.

- A confident answer is **not** the same as a correct or relevant one.
- The categories a model can output are a form of **power**: they decide what is
  nameable and what is invisible.
- A model trained on general web images may be useless for a specific community's
  plants, animals, tools, or land — even though it "works" on benchmarks.

**Discuss:** who decided the 1,000 ImageNet categories, and whose world do they
reflect? What would a category list designed *by your community* include instead?


## 🟡 Part 2 — Zero-shot classification: you choose the categories

What if *we* could decide the categories? **CLIP** is a model trained to match
images with text descriptions. With "zero-shot" classification we hand it a list
of **candidate labels in plain language**, and it scores how well the image
matches each one — no retraining needed.

This puts the category list back in *your* hands. That is powerful — and it also
means the result depends entirely on how you phrase the options.


In [ ]:
zero_shot = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=PIPELINE_DEVICE,
)

print("Loaded zero-shot model:", zero_shot.model.name_or_path)


### ✍️ Your turn — write the candidate labels

Describe, in natural language, the things this image *could* be. Good zero-shot
labels are short descriptive phrases, e.g. `"a photo of a healthy leaf"`.


In [ ]:
# TODO: replace None with a list of 3-5 descriptive label strings.
# Hint: think about the three real categories you printed earlier.
candidate_labels = None  # <-- your code here

# This line will fail until you fill in candidate_labels above.
results = zero_shot(image, candidate_labels=candidate_labels)
for r in results:
    print(f"{r['label']:<45} score={r['score']:.3f}")


#### ✅ Solution


In [ ]:
candidate_labels = [
    "a healthy bean leaf",
    "a bean leaf with angular leaf spot disease",
    "a bean leaf with rust disease",
]

results = zero_shot(image, candidate_labels=candidate_labels)
print("True label:", class_names[example["labels"]])
print()
for r in results:
    print(f"{r['label']:<45} score={r['score']:.3f}")


### ▶️ Play with it — the wording changes the answer

Run the cell below, then **edit the labels** and rerun. Try:

- very generic labels (`"a leaf"`, `"a plant"`, `"a rock"`)
- labels in a language other than English
- adding an irrelevant label (`"a car"`) and see how scores shift
- removing the correct label entirely — what does the model do then?


In [ ]:
def classify_with_labels(img, labels):
    scores = zero_shot(img, candidate_labels=labels)
    for r in scores:
        print(f"{r['label']:<45} score={r['score']:.3f}")
    print("-" * 60)
    return scores

# Edit this list freely and rerun the cell.
my_labels = [
    "a healthy plant leaf",
    "a diseased plant leaf",
    "a car",
]

for i in range(3):
    print("Image", i, "| true label:", class_names[beans[i]["labels"]])
    classify_with_labels(beans[i]["image"], my_labels)


### 🧭 Critical reflection: who writes the labels?

Zero-shot models *feel* neutral, but every result is shaped by the label list and
by the text–image pairs CLIP learned from (largely English-language internet data).

- The model can only ever choose from the options **you** provide. Whoever writes the label list holds real power over the outcome.
- CLIP often does worse on concepts, languages and places that are underrepresented online.
- **Representation gap:** if a community's terms, languages, or categories were not in the training data, the model is effectively guessing.


## 🧪 More experiments — push the models and watch them break

Real photos are messy: odd angles, shadows, blur, bad light. The next two experiments probe **how robust** a model is and **how much the choice of model matters** — both are things you need to know *before* trusting a model in the field.


### 🧪 Experiment 1 — robustness to messy field photos

**Why this experiment?** A model can score well on clean dataset images yet fail on a
farmer's quick phone snapshot. Here we distort one image (rotate, dim, brighten, blur)
and watch whether the prediction holds. If small, harmless changes flip the answer,
the model is *brittle* — a serious problem for real-world deployment.


In [ ]:
from PIL import ImageEnhance, ImageFilter

def perturb(img, kind):
    if kind == "rotate":  return img.rotate(25)
    if kind == "dark":    return ImageEnhance.Brightness(img).enhance(0.5)
    if kind == "bright":  return ImageEnhance.Brightness(img).enhance(1.7)
    if kind == "blur":    return img.filter(ImageFilter.GaussianBlur(3))
    return img

test_labels = [
    "a healthy bean leaf",
    "a bean leaf with angular leaf spot disease",
    "a bean leaf with rust disease",
]

print("True label:", class_names[example["labels"]])
for kind in ["original", "rotate", "dark", "bright", "blur"]:
    top = zero_shot(perturb(image, kind), candidate_labels=test_labels)[0]
    print(f"  {kind:<9} -> {top['label']:<45} ({top['score']:.2f})")


### ▶️ Play with it — does a different architecture see better?

**Why this experiment?** "The model" is a *choice*, not a given. Swapping a Vision
Transformer for a convolutional ResNet (both ImageNet-trained) shows that different
architectures make different mistakes. Try other model names from the Hub and
compare — newer or bigger is not always better for *your* data.


In [ ]:
# Compare two ImageNet backbones on the same leaf image.
# (The first run downloads the model; this cell is safe to edit and rerun.)
other_backbone = "microsoft/resnet-50"   # <-- try e.g. "facebook/convnext-tiny-224"
other_classifier = pipeline("image-classification", model=other_backbone, device=PIPELINE_DEVICE)

print("ViT (google/vit-base-patch16-224) top-1:")
print("  ", imagenet_classifier(image, top_k=1)[0])
print(other_backbone, "top-1:")
print("  ", other_classifier(image, top_k=1)[0])
print()
print("Neither was trained to know bean-leaf diseases — compare *how* they are wrong.")


## 🔴 Optional / Advanced — fine-tune your own vision model

Pretrained and zero-shot models are great starting points, but the strongest
results on a *specific* community task usually come from **fine-tuning**: taking a
pretrained backbone and training it a little more on your own labelled images.

Below we fine-tune a ViT on the beans dataset. This needs a **GPU** and a few
minutes. (Notebook 02 fine-tunes a *text* model with the same Hugging Face
`Trainer` — notice how similar the recipe is across modalities.)


In [ ]:
# !pip -q install transformers[torch] datasets scikit-learn

from transformers import AutoImageProcessor, AutoModelForImageClassification
from transformers import TrainingArguments, Trainer
from torchvision.transforms import Compose, Resize, CenterCrop, RandomResizedCrop, ToTensor, Normalize
from sklearn.metrics import accuracy_score
import numpy as np
import torch

# A backbone meant for fine-tuning (pretrained on ImageNet-21k, no final classes).
checkpoint = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(checkpoint)
print("Image size expected by the model:", processor.size)


In [ ]:
# Build label <-> id mappings from the dataset itself.
full = load_dataset("AI-Lab-Makerere/beans")
labels = full["train"].features["labels"].names
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in enumerate(labels)}
print(id2label)


In [ ]:
# Image preprocessing. We resize/crop to the model's expected size and normalise
# using the SAME statistics the backbone was trained with.
if "height" in processor.size:
    size = (processor.size["height"], processor.size["width"])
    crop = size[0]
else:
    crop = processor.size["shortest_edge"]
    size = (crop, crop)

normalize = Normalize(mean=processor.image_mean, std=processor.image_std)

train_transforms = Compose([RandomResizedCrop(crop), ToTensor(), normalize])
eval_transforms = Compose([Resize(size), CenterCrop(crop), ToTensor(), normalize])

def apply_train(batch):
    batch["pixel_values"] = [train_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

def apply_eval(batch):
    batch["pixel_values"] = [eval_transforms(img.convert("RGB")) for img in batch["image"]]
    return batch

train_ds = full["train"].with_transform(apply_train)
val_ds = full["validation"].with_transform(apply_eval)
print("Train images:", len(train_ds), "| Validation images:", len(val_ds))


In [ ]:
# Collate function: stack the processed tensors into a batch.
def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["labels"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(gold, preds)}

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


### ✍️ Your turn — set the training hyperparameters

Fill in a few key knobs. Sensible starting points: learning rate around `2e-4`,
batch size `16`, and `3` epochs. You can tune these afterwards.


In [ ]:
# TODO: replace each ... with a value.
LEARNING_RATE = ...   # try 2e-4
BATCH_SIZE = ...      # try 16
NUM_EPOCHS = ...      # try 3

args = TrainingArguments(
    output_dir="vit-beans",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
)


#### ✅ Solution


In [ ]:
def build_training_args(**kwargs):
    # Older transformers used `evaluation_strategy`; newer use `eval_strategy`.
    try:
        return TrainingArguments(eval_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kwargs)

args = build_training_args(
    output_dir="vit-beans",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="no",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,  # keep the 'image' column for our transform
    report_to="none",
)


In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())


In [ ]:
# ▶️ Try your fine-tuned model on a validation image.
model.eval()
test_image = full["validation"][5]["image"]
inputs = {"pixel_values": eval_transforms(test_image.convert("RGB")).unsqueeze(0).to(model.device)}
with torch.no_grad():
    logits = model(**inputs).logits
predicted = id2label[int(logits.argmax(-1))]

plt.figure(figsize=(4, 4))
plt.imshow(test_image)
plt.axis("off")
plt.title("Predicted: " + predicted + " | True: " + id2label[full["validation"][5]["labels"]])
plt.show()


## 🔴 Optional / Advanced — build your own CNN from scratch (and see why pretraining wins)

**Why this experiment?** Everything above reused a model that was *pretrained* on
millions of images. Here you build a small **Convolutional Neural Network** from
scratch — the architecture family from Workshop 1 — and train it on beans *alone*.

Then compare its accuracy to the fine-tuned ViT. The from-scratch CNN usually lands
far lower — not because CNNs are bad, but because it never saw the millions of images
that gave the ViT its head start. **That gap is the entire argument for transfer
learning**, and it matters most for communities with little labelled data.


In [ ]:
# A small, self-contained image pipeline. Smaller images => faster from-scratch training.
from datasets import load_dataset
from torchvision.transforms import Compose, Resize, ToTensor
from torch.utils.data import DataLoader
import torch
from torch import nn

beans_full = load_dataset("AI-Lab-Makerere/beans")
cnn_class_names = beans_full["train"].features["labels"].names

IMG_SIZE = 96
cnn_tf = Compose([Resize((IMG_SIZE, IMG_SIZE)), ToTensor()])

def cnn_transform(batch):
    batch["pixel_values"] = [cnn_tf(im.convert("RGB")) for im in batch["image"]]
    return batch

def cnn_collate(batch):
    x = torch.stack([b["pixel_values"] for b in batch])
    y = torch.tensor([b["labels"] for b in batch])
    return x, y

cnn_train_loader = DataLoader(
    beans_full["train"].with_transform(cnn_transform),
    batch_size=32, shuffle=True, collate_fn=cnn_collate,
)
cnn_val_loader = DataLoader(
    beans_full["validation"].with_transform(cnn_transform),
    batch_size=32, collate_fn=cnn_collate,
)
print("Batches:", len(cnn_train_loader), "train,", len(cnn_val_loader), "val")


### ✍️ Your turn — design the convolutional feature extractor

Fill in a stack of convolution blocks. A common pattern repeats
`Conv2d -> ReLU -> MaxPool2d`, growing the channels (e.g. 3 -> 16 -> 32 -> 64).
Each `MaxPool2d(2)` halves the height and width.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # TODO: build a nn.Sequential of Conv2d/ReLU/MaxPool2d blocks.
        # Start from 3 input channels and end with 64 output channels.
        self.features = ...
        self.pool = nn.AdaptiveAvgPool2d(1)   # collapse to (batch, 64, 1, 1)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

# Should print torch.Size([2, 3]) once you have filled in the feature extractor.
print(SmallCNN(3)(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape)


#### ✅ Solution


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 96 -> 48
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 48 -> 24
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 24 -> 12
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

print(SmallCNN(3)(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape)  # expect (2, 3)


In [ ]:
def train_cnn(model, epochs=5, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        for xb, yb in cnn_train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        # quick validation pass
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in cnn_val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                correct += (model(xb).argmax(-1) == yb).sum().item()
                total += len(yb)
        print(f"Epoch {epoch + 1}/{epochs}  val accuracy: {correct / total:.3f}")
    return model

torch.manual_seed(SEED)
cnn_model = train_cnn(SmallCNN(len(cnn_class_names)), epochs=5, lr=1e-3)


### 🧭 Critical reflection: the value of what you start from

- Note the CNN's best validation accuracy and compare it to the fine-tuned ViT above.
- Pretraining data, like compute, is a form of **power** that is unevenly distributed.
  Communities that fine-tune existing models inherit both the strengths *and* the
  blind spots of whoever did the pretraining.
- **Try it:** train the CNN for many more epochs. Does train accuracy keep climbing
  while validation accuracy stalls? That gap is *overfitting* — a constant risk when
  data is scarce, exactly the situation many community projects face.
